In [ ]:
import json
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter 
import numpy as np
from enum import Enum
import os

In [ ]:
algorithmNameList = ['ParEGO', 'MParEGO', 'Sobol', 'Random']
dimensionsList = [2, 3, 5, 10, 20]
read_ecdf_data = True
runs_data_path = '../outputs'
ecdf_data_path = 'ecdf_data'
ecdf_plot_path = 'bbob_plots'
budget = 1000

In [ ]:
functionID_folder_dic = {
    1: '1-separable_1-separable',
    2: '1-separable_1-separable',
    3: '1-separable_2-moderate',
    4: '1-separable_2-moderate',
    5: '1-separable_3-ill-conditioned',
    6: '1-separable_3-ill-conditioned',
    7: '1-separable_4-multi-modal',
    8: '1-separable_4-multi-modal',
    9: '1-separable_5-weakly-structured',
    10: '1-separable_5-weakly-structured',
    11: '1-separable_1-separable',
    12: '1-separable_2-moderate',
    13: '1-separable_2-moderate',
    14: '1-separable_3-ill-conditioned',
    15: '1-separable_3-ill-conditioned',
    16: '1-separable_4-multi-modal',
    17: '1-separable_4-multi-modal',
    18: '1-separable_5-weakly-structured',
    19: '1-separable_5-weakly-structured',
    20: '2-moderate_2-moderate',
    21: '2-moderate_2-moderate',
    22: '2-moderate_3-ill-conditioned',
    23: '2-moderate_3-ill-conditioned',
    24: '2-moderate_4-multi-modal',
    25: '2-moderate_4-multi-modal',
    26: '2-moderate_5-weakly-structured',
    27: '2-moderate_5-weakly-structured',
    28: '2-moderate_2-moderate',
    29: '2-moderate_3-ill-conditioned',
    30: '2-moderate_3-ill-conditioned',
    31: '2-moderate_4-multi-modal',
    32: '2-moderate_4-multi-modal',
    33: '2-moderate_5-weakly-structured',
    34: '2-moderate_5-weakly-structured',
    35: '3-ill-conditioned_3-ill-conditioned',
    36: '3-ill-conditioned_3-ill-conditioned',
    37: '3-ill-conditioned_4-multi-modal',
    38: '3-ill-conditioned_4-multi-modal',
    39: '3-ill-conditioned_5-weakly-structured',
    40: '3-ill-conditioned_5-weakly-structured',
    41: '3-ill-conditioned_3-ill-conditioned',
    42: '3-ill-conditioned_4-multi-modal',
    43: '3-ill-conditioned_4-multi-modal',
    44: '3-ill-conditioned_5-weakly-structured',
    45: '3-ill-conditioned_5-weakly-structured',
    46: '4-multi-modal_4-multi-modal',
    47: '4-multi-modal_4-multi-modal',
    48: '4-multi-modal_5-weakly-structured',
    49: '4-multi-modal_5-weakly-structured',
    50: '4-multi-modal_4-multi-modal',
    51: '4-multi-modal_5-weakly-structured',
    52: '4-multi-modal_5-weakly-structured',
    53: '5-weakly-structured_5-weakly-structured',
    54: '5-weakly-structured_5-weakly-structured',
    55: '5-weakly-structured_5-weakly-structured'
}

# Read files functions

In [ ]:
def read_eval_json_file(filename):
    f = open(filename)
    data = json.load(f)
    f.close()
    return data['Evaluations']

def read_pop_json_file(filename):
    f = open(filename)
    data = json.load(f)
    f.close()
    return data['Populations'][0]

def read_eval_file(functionID, nInputs, instanceID, algorithmName='ParEGO'):
    algo_runs_path = os.path.join(runs_data_path, algorithmName)
    algo_d_path = os.path.join(algo_runs_path, 'D' + str(nInputs))
    algo_f_path = os.path.join(algo_d_path, 'runD' + str(nInputs) + 'F' + str(functionID))
    filename = 'evaluations_' + algorithmName + '_BBOB-BIOBJ_F' + str(functionID) + '_D' + str(nInputs) + '_I' + str(instanceID) + '_S' + str(instanceID) + '.json'
    jfile = os.path.join(algo_f_path, filename)
    return read_eval_json_file(jfile)

def read_pop_file(functionID, nInputs, instanceID, algorithmName='ParEGO'):
    algo_runs_path = os.path.join(runs_data_path, algorithmName)
    algo_d_path = os.path.join(algo_runs_path, 'D' + str(nInputs))
    algo_f_path = os.path.join(algo_d_path, 'runD' + str(nInputs) + 'F' + str(functionID))
    filename = 'population_' + algorithmName + '_BBOB-BIOBJ_F' + str(functionID) + '_D' + str(nInputs) + '_I' + str(instanceID) + '_S' + str(instanceID) + '.json'
    jfile = os.path.join(algo_f_path, filename)
    return read_pop_json_file(jfile)
    
def read_ideal_vector(functionID, nInputs, instanceID, algorithmName):
    pop = read_pop_file(functionID, nInputs, instanceID)  # use the default seed and default algorithm
    return pop['Ideal Vector']

def read_nadir_vector(functionID, nInputs, instanceID, algorithmName):
    pop = read_pop_file(functionID, nInputs, instanceID)  # use the default seed and default algorithm
    return pop['Nadir Vector']

def read_best_hyp(functionID, nInputs, instanceID):
    pop = read_pop_file(functionID, nInputs, instanceID)  # use the default seed and default algorithm
    return pop['Best Hypervolume']

def read_data(file_path):
    # Initialize a list to store the data
    data = []
    # Open the file and read it line by line
    with open(file_path, 'r') as file:
        for line in file:
            # Skip lines that start with '%'
            if line.startswith('%'):
                continue
            # Split the line by tabs and convert the values to float
            values = line.strip().split('\t')
            if len(values) == 2:  # Ensure there are exactly two columns
                try:
                    data.append([int(values[0]), float(values[1])])
                except ValueError:
                    print(f"Skipping invalid line: {line.strip()}")
    # Convert the list to a NumPy array
    return np.array(data)

def numpy_fill(arr):
    '''Solution provided by Divakar.'''
    mask = np.isnan(arr)
    idx = np.where(~mask,np.arange(mask.size),0)
    np.maximum.accumulate(idx, out=idx)
    out = arr[idx]
    return out

def read_tdat_file(functionID, nInputs, algorithmName='TPB_b50'):
    group_str = functionID_folder_dic[functionID]
    data_path = os.path.join(runs_data_path, algorithmName)
    data_path = os.path.join(data_path, group_str)
    filename = 'bbob-biobj_f' + str(functionID).rjust(2,'0') + '_d' + str(nInputs).rjust(2,'0') + '_hyp.tdat'
    tfile = os.path.join(data_path, filename)
    if os.path.exists(tfile):
        return read_data(tfile)
    return tfile

def read_icoco(functionID, nInputs, instanceID, algorithmName):
    if (algorithmName=='ParEGO') or (algorithmName=='MParEGO') or (algorithmName=='CParEGO') or (algorithmName=='HParEGO') or (algorithmName=='Sobol') or (algorithmName=='Random'):  # Add Tigon algorithms here
        pop = read_pop_file(functionID, nInputs, instanceID, algorithmName)
        a = np.array(pop['ICoco'])
        return a
    else:  # Add the Algorithms that have been run with the COCO observer
        # 1. select the instanceID
        data_np = read_tdat_file(functionID, nInputs, algorithmName)
        indices = np.where(data_np[:,0].astype(int) == 1)[0]
        indices = np.append(indices, data_np.size)
        vinit = int(indices[instanceID-1])
        vend = int(indices[instanceID])
        data_sel_np = data_np[vinit:vend,]
        # 2. fill the missing function evaluations
        asize = int(data_sel_np[-1,0])
        a = np.empty(asize)
        a[:] = np.nan
        indices_int = data_sel_np[:,0].astype(int)
        a[indices_int-1] = np.copy(data_sel_np[:,1])
        at = numpy_fill(a)
        # 3. collect only up-to maximum budget of function evaluations
        tbudget = min(budget, at.size)
        att = at[:tbudget]
        # 4. convert to Icoco by subtracting the best indicator value to hypervolume detemined values
        bhyp = read_best_hyp(functionID, nInputs, instanceID)
        att[((att-bhyp)<0)] -= bhyp
        return att

In [ ]:
# read_nadir_vector(54, 3, 1, 'ParEGO')
# read_best_hyp(54, 3, 1)
read_icoco(54, 3, 1, 'ParEGO')
read_best_hyp(54, 3, 1)

# ICoco functions

In [ ]:
def icoco_instances(functionID, nInputs, algorithmName):  # icoco across instances
    nInstances = 15
    icoco_list = [read_icoco(functionID, nInputs, i+1, algorithmName) for i in range(nInstances)]
    icoco_np = np.array(icoco_list)  # 15 x 1000
    return icoco_np

def icoco_instance(functionID, nInputs, algorithmName, instanceID):
    icoco = read_icoco(functionID, nInputs, instanceID, algorithmName)
    icoco_np = np.array(icoco)  # 1000 x 1
    return icoco_np

def icoco_diff_instances(functionID, nInputs, algorithmName):  # icoco across instances
    nInstances = 15
    icoco_list = [read_icoco(functionID, nInputs, i+1, algorithmName) for i in range(nInstances)]
    icoco_np = np.array(icoco_list)  # 15 x 1000
    besthv_list = [read_best_hyp(functionID, nInputs, i+1) for i in range(nInstances)]
    besthv_np = -np.array(besthv_list)  # 15 x 1
    icoco_diff_np = np.transpose(icoco_np)-besthv_np  # 1000 x 15
    hyp = np.transpose(icoco_diff_np)                 # 15 x 1000
    return hyp

In [ ]:
icoco_instances(54, 3, 'ParEGO')

# Target functions

In [ ]:
class PrecisionTargetsType(Enum):
    """
    Enum class for precision targets
    """
    EC_POS = 1
    EC_ALL = 2
    BUDGET = 3
    OMS = 4

class Targets:

    def __init__(self):
        self._precision_target_type = PrecisionTargetsType.EC_POS
        self.update_precision_targets()

    def change_precision_targets(self, _precision_target_type):
        if not isinstance(_precision_target_type, PrecisionTargetsType):
            raise ValueError("precision_target_type must be an instance of the class PrecisionTargetsType")
        self._precision_target_type = _precision_target_type
        self.update_precision_targets()

    def update_precision_targets(self):
        if self._precision_target_type == PrecisionTargetsType.EC_POS:
            self.precision_targets_np = self.precision_targets_ec_pos()
        elif self._precision_target_type == PrecisionTargetsType.EC_ALL:
            self.precision_targets_np = self.precision_targets_ec_all()
        elif self._precision_target_type == PrecisionTargetsType.BUDGET:
            self.precision_targets_np = None
        elif self._precision_target_type == PrecisionTargetsType.OMS:
            self.precision_targets_np = self.precision_targets_oms()

    def prevision_targets_str(self):
        return self._precision_target_type.name.lower()
            
    def targets(self, functionID, nInputs, instanceID):
        if (self._precision_target_type == PrecisionTargetsType.EC_POS) or (self._precision_target_type == PrecisionTargetsType.EC_ALL) or (self._precision_target_type == PrecisionTargetsType.OMS):
            refhv = -read_best_hyp(functionID, nInputs, instanceID)
            targets_np = refhv + self.precision_targets_np
        elif self._precision_target_type == PrecisionTargetsType.BUDGET:
            last_icoco_list = []
            first_icoco_list = []
            starget = int(math.ceil(0.5*nInputs))
            ftarget = int(math.ceil(50*nInputs))
            for algorithmName in algorithmNameList:
                data = read_icoco(functionID, nInputs, instanceID, algorithmName)
                st = min(starget, data.size)
                ft = min(ftarget, data.size)
                data_st = data[st-1]
                data_ft = data[ft-1]
                first_icoco_list.append(data_st)
                last_icoco_list.append(data_ft)
            href_best = min(last_icoco_list)
            href_worst = min(first_icoco_list)
            besthv = -read_best_hyp(functionID, nInputs, instanceID)
            targets_np = np.geomspace(href_best - besthv, href_worst - besthv, num=self.number_precision_targets())
            targets_np = targets_np - (-besthv)
            targets_np = np.flip(targets_np)
        return targets_np

    def precision_targets(self):
        return self.precision_targets_np

    def number_precision_targets(self):
        if self._precision_target_type == PrecisionTargetsType.EC_POS:
            return 52
        if self._precision_target_type == PrecisionTargetsType.EC_ALL:
            return 58
        if self._precision_target_type == PrecisionTargetsType.BUDGET:
            return 31
        if self._precision_target_type == PrecisionTargetsType.OMS:
            return 51

    def targets_msg(self):
        if self._precision_target_type == PrecisionTargetsType.EC_POS:
            return str(self.number_precision_targets()) + ' targets: 1..0'
        if self._precision_target_type == PrecisionTargetsType.EC_ALL:
            return str(self.number_precision_targets()) + ' targets: 1..-1.0e-4'
        if self._precision_target_type == PrecisionTargetsType.BUDGET:
            return str(self.number_precision_targets()) + ' targets: 0.5..50'
        if self._precision_target_type == PrecisionTargetsType.OMS:
            return str(self.number_precision_targets()) + ' targets: 100..1.0e-8'

    def precision_targets_oms(self):
        return np.flip(np.geomspace(10**(-8), 100, self.number_precision_targets()))
    
    def precision_targets_ec_pos(self):  # evolutionary computation paper precision targets
        # 58 targets
        precision_targets_neg = [-10**(-(4+(i/10)*2)) for i in range(6)]  # better than optima (6)
        precision_targets_pos = [10**(-i/10) for i in range(58-6)]  # worst than optima (52)
        precision_targets = precision_targets_neg + list(reversed(precision_targets_pos))
        precision_targets = list(reversed(precision_targets))
        precision_targets_np = np.array(precision_targets)
        return precision_targets_np[:52]  # only the positive precision targets
    
    def precision_targets_ec_all(self):  # evolutionary computation paper precision targets
        # 58 targets
        precision_targets_neg = [-10**(-(4+(i/10)*2)) for i in range(6)]  # better than optima (6)
        precision_targets_pos = [10**(-i/10) for i in range(58-6)]  # worst than optima (52)
        precision_targets = precision_targets_neg + list(reversed(precision_targets_pos))
        precision_targets = list(reversed(precision_targets))
        precision_targets_np = np.array(precision_targets)
        return precision_targets_np

In [ ]:
targets_test = Targets()
targets_test.change_precision_targets(PrecisionTargetsType.OMS)
targets_test.targets(54, 3, 1)
# targets_test.precision_targets()
# targets_test.number_precision_targets()
# targets_test.targets_msg()
# targets_test.targets(37, 5, 1)
# targets_test.prevision_targets_str()

In [ ]:
targets_test = Targets()
targets_test.change_precision_targets(PrecisionTargetsType.BUDGET)
targets_test.targets(54, 3, 11)

In [ ]:
# icoco_instances(37, 5, "MParEGO")[0]

# ECDF generation functions

In [ ]:
# ECDF generation functions
def ecdf_individual_function(functionID, nInputs, algorithmName, targets_func):
    number_instances = 15
    number_precision_targets = targets_func(functionID=1, nInputs=nInputs, instanceID=1).size
    
    hyp = icoco_instances(functionID, nInputs, algorithmName)  # 15 x 1000
    nn, number_function_evaluations = hyp.shape
    xadd_np = np.full(number_function_evaluations, 0)
    
    for instanceID in range(number_instances):
        xs = []
        targets_np = targets_func(functionID=functionID, nInputs=nInputs, instanceID=instanceID+1)
        for icoco in hyp[instanceID]:
            count = 0
            for ref in targets_np:
                if (icoco < ref):
                    count = count + 1
            xs.append(count)
        xadd_np = xadd_np + np.array(xs)

    xsp = xadd_np / (number_instances * number_precision_targets)
    return xsp

def ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func):
    number_functions = len(functionIDList)
    number_instances = 15
    number_precision_targets = targets_func(functionID=1, nInputs=nInputs, instanceID=1).size
    
    hyp_ref = icoco_instances(1, nInputs, algorithmName)  # 15 x 1000
    nn, number_function_evaluations = hyp_ref.shape
    xadd_np = np.full(number_function_evaluations, 0)
    
    for functionID in functionIDList:
        hyp = icoco_instances(functionID, nInputs, algorithmName)  # 15 x 1000
        for instanceID in range(number_instances):
            xs = []
            targets_np = targets_func(functionID=functionID, nInputs=nInputs, instanceID=instanceID+1)
            for icoco in hyp[instanceID]:
                count = 0
                for ref in targets_np:
                    if (icoco < ref):
                        count = count + 1
                xs.append(count)
            xadd_np = xadd_np + np.array(xs)
    xsp = xadd_np / (number_instances * number_precision_targets * number_functions)
    return xsp

def ecdf_all_functions(nInputs, algorithmName, targets_func):
    functionIDList = list(range(1,56))
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_ss_functions(nInputs, algorithmName, targets_func):
    functionIDList = [1, 2, 11]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_smo_functions(nInputs, algorithmName, targets_func):
    functionIDList = [3, 4, 12, 13]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_si_functions(nInputs, algorithmName, targets_func):
    functionIDList = [5, 6, 14, 15]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_smu_functions(nInputs, algorithmName, targets_func):
    functionIDList = [7, 8, 16, 17]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_sw_functions(nInputs, algorithmName, targets_func):
    functionIDList = [9, 10, 18, 19]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_momo_functions(nInputs, algorithmName, targets_func):
    functionIDList = [20, 21, 28]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_moi_functions(nInputs, algorithmName, targets_func):
    functionIDList = [22, 23, 29, 30]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_momu_functions(nInputs, algorithmName, targets_func):
    functionIDList = [24, 25, 31, 32]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_mow_functions(nInputs, algorithmName, targets_func):
    functionIDList = [26, 27, 33, 34]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_ii_functions(nInputs, algorithmName, targets_func):
    functionIDList = [35, 36, 41]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_imu_functions(nInputs, algorithmName, targets_func):
    functionIDList = [37, 38, 42, 43]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_iw_functions(nInputs, algorithmName, targets_func):
    functionIDList = [39, 40, 44, 45]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_mumu_functions(nInputs, algorithmName, targets_func):
    functionIDList = [46, 47, 50]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_muw_functions(nInputs, algorithmName, targets_func):
    functionIDList = [48, 49, 51, 52]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

def ecdf_ww_functions(nInputs, algorithmName, targets_func):
    functionIDList = [53, 54, 55]
    return ecdf_user_functions(functionIDList, nInputs, algorithmName, targets_func)

# ECDF plot functions

In [ ]:
# Function groups plots of the bbob-biobj
def ecdf_ss_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f1, f2, f11, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_ss_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='separable-separable', filename=filename)

def ecdf_ss_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f1, f2, f11\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_ss_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='separable-separable', filename=filename)

def ecdf_smo_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f3, f4, f12, f13, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_smo_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='separable-moderate', filename=filename)

def ecdf_smo_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f3, f4, f12, f13\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_smo_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='separable-moderate', filename=filename)

def ecdf_si_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f5, f6, f14, f15, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_si_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='separable-ill-cond.', filename=filename)

def ecdf_si_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f5, f6, f14, f15\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_si_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='separable-ill-cond.', filename=filename)
    
def ecdf_smu_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f7, f8, f16, f17, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_smu_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='separable-multimodal', filename=filename)

def ecdf_smu_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f7, f8, f16, f17\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_smu_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='separable-multimodal', filename=filename)

def ecdf_sw_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f9, f10, f18, f19, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_sw_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='separable-weakstructure', filename=filename)

def ecdf_sw_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f9, f10, f18, f19\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_sw_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='separable-weakstructure', filename=filename)

def ecdf_momo_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f20, f21, f28, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_momo_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='moderate-moderate', filename=filename)

def ecdf_momo_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f20, f21, f28\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_momo_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='moderate-moderate', filename=filename)

def ecdf_moi_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f22, f23, f29, f30, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_moi_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='moderate-ill-cond.', filename=filename)

def ecdf_moi_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f22, f23, f29, f30\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_moi_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='moderate-ill-cond.', filename=filename)

def ecdf_momu_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f24, f25, f31, f32, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_momu_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='moderate-multimodal', filename=filename)

def ecdf_momu_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f24, f25, f31, f32\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_momu_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='moderate-multimodal', filename=filename)

def ecdf_mow_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f26, f27, f33, f34, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_mow_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='moderate-weakstructure', filename=filename)

def ecdf_mow_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f26, f27, f33, f34\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_mow_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='moderate-weakstructure', filename=filename)

def ecdf_ii_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f35, f36, f41, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_ii_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='ill-cond.-ill-cond.', filename=filename)

def ecdf_ii_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f35, f36, f41\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_ii_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='ill-cond.-ill-cond.', filename=filename)

def ecdf_imu_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f37, f38, f42, f43, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_imu_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='ill-cond.-multimodal', filename=filename)

def ecdf_imu_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f37, f38, f42, f43\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_imu_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='ill-cond.-multimodal', filename=filename)

def ecdf_iw_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f39, f40, f44, f45, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_iw_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='ill-cond.-weakstructure', filename=filename)

def ecdf_iw_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f39, f40, f44, f45\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_iw_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='ill-cond.-weakstructure', filename=filename)

def ecdf_mumu_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f46, f47, f50, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_mumu_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='multimodal-multimodal', filename=filename)

def ecdf_mumu_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f46, f47, f50\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_mumu_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='multimodal-multimodal', filename=filename)

def ecdf_muw_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f48, f49, f51, f52, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_muw_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='multimodal-weakstructure', filename=filename)

def ecdf_muw_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f48, f49, f51, f52\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_muw_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='multimodal-weakstructure', filename=filename)

def ecdf_ww_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f53, f54, f55, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'group_plots/BBOB-BIOBJ_ECDF_group_ww_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='weakstructure-weakstructure', filename=filename)

def ecdf_ww_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f53, f54, f55\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/group_plots/BBOB-BIOBJ_ECDF_group_ww_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='weakstructure-weakstructure', filename=filename)

In [ ]:
# Function individual plots of the bbob-biobj
def ecdf_f1_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f1, ' + str(nInputs) + '-D\n' +  targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f1_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='1 Sphere/Sphere', filename=filename)

def ecdf_f1_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f1\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f1_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='1 Sphere/Sphere', filename=filename)

def ecdf_f2_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f2, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f2_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='2 Sphere/sep. Ellipsoid', filename=filename)

def ecdf_f2_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f2\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f2_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='2 Sphere/sep. Ellipsoid', filename=filename)

def ecdf_f3_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f3, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f3_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='3 Sphere/Attr. sector', filename=filename)

def ecdf_f3_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f3\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f3_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='3 Sphere/Attr. sector', filename=filename)

def ecdf_f4_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f4, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f4_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='4 Sphere/Rosenbrock', filename=filename)

def ecdf_f4_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f4\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f4_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='4 Sphere/Rosenbrock', filename=filename)

def ecdf_f5_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f5, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f5_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='5 Sphere/Sharp ridge', filename=filename)

def ecdf_f5_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f5\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f5_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='5 Sphere/Sharp ridge', filename=filename)

def ecdf_f6_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f6, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f6_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='6 Sphere/Different Powers', filename=filename)

def ecdf_f6_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f6\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f6_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='6 Sphere/Different Powers', filename=filename)

def ecdf_f7_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f7, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f7_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='7 Sphere/Rastrigin', filename=filename)

def ecdf_f7_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f7\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f7_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='7 Sphere/Rastrigin', filename=filename)

def ecdf_f8_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f8, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f8_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='8 Sphere/Schaffer F7', filename=filename)

def ecdf_f8_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f8\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f8_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='8 Sphere/Schaffer F7', filename=filename)

def ecdf_f9_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f9, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f9_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='9 Sphere/Schwefel', filename=filename)

def ecdf_f9_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f9\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f9_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='9 Sphere/Schwefel', filename=filename)

def ecdf_f10_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f10, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f10_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='10 Sphere/Gallagher 101', filename=filename)

def ecdf_f10_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f10\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f10_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='10 Sphere/Gallagher 101', filename=filename)

def ecdf_f11_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f11, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f11_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='11 sep. Ellipsoid/sep. Elli.', filename=filename)

def ecdf_f11_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f11\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f11_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='11 sep. Ellipsoid/sep. Elli.', filename=filename)

def ecdf_f12_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f12, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f12_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='12 sep. Ellipsoid/Attr. sector', filename=filename)

def ecdf_f12_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f12\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f12_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='12 sep. Ellipsoid/Attr. sector', filename=filename)

def ecdf_f13_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f13, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f13_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='13 sep. Ellipsoid/Rosenbrock', filename=filename)

def ecdf_f13_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f13\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f13_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='13 sep. Ellipsoid/Rosenbrock', filename=filename)

def ecdf_f14_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f14, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f14_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='14 sep. Ellipsoid/Sharp ridge', filename=filename)

def ecdf_f14_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f14\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f14_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='14 sep. Ellipsoid/Sharp ridge', filename=filename)

def ecdf_f15_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f15, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f15_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='15 sep. Ellipsoid/Diff. Powers', filename=filename)

def ecdf_f15_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f15\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f15_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='15 sep. Ellipsoid/Diff. Powers', filename=filename)

def ecdf_f16_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f16, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f16_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='16 sep. Ellipsoid/Rastrigin', filename=filename)

def ecdf_f16_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f16\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f16_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='16 sep. Ellipsoid/Rastrigin', filename=filename)

def ecdf_f17_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f17, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f17_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='17 sep. Ellipsoid/Schaffer F7', filename=filename)

def ecdf_f17_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f17\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f17_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='17 sep. Ellipsoid/Schaffer F7', filename=filename)

def ecdf_f18_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f18, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f18_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='18 sep. Ellipsoid/Schwefel', filename=filename)

def ecdf_f18_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f18\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f18_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='18 sep. Ellipsoid/Schwefel', filename=filename)

def ecdf_f19_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f19, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f19_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='19 sep. Elli./Gallagher 101', filename=filename)

def ecdf_f19_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f19\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f19_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='19 sep. Elli./Gallagher 101', filename=filename)

def ecdf_f20_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f20, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f20_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='20 Attr. sector/Attr. sector', filename=filename)

def ecdf_f20_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f20\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f20_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='20 Attr. sector/Attr. sector', filename=filename)

def ecdf_f21_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f21, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f21_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='21 Attr. sector/Rosenbrock', filename=filename)

def ecdf_f21_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f21\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f21_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='21 Attr. sector/Rosenbrock', filename=filename)

def ecdf_f22_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f22, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f22_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='22 Attr. sector/Sharp ridge', filename=filename)

def ecdf_f22_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f22\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f22_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='22 Attr. sector/Sharp ridge', filename=filename)

def ecdf_f23_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f23, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f23_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='23 Attr. sector/Diff. Powers', filename=filename)

def ecdf_f23_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f23\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f23_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='23 Attr. sector/Diff. Powers', filename=filename)

def ecdf_f24_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f24, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f24_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='24 Attr. sector/Rastrigin', filename=filename)

def ecdf_f24_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f24\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f24_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='24 Attr. sector/Rastrigin', filename=filename)

def ecdf_f25_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f25, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f25_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='25 Attr. sector/Schaffer F7', filename=filename)

def ecdf_f25_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f25\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f25_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='25 Attr. sector/Schaffer F7', filename=filename)

def ecdf_f26_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f26, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f26_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='26 Attr. sector/Schwefel', filename=filename)

def ecdf_f26_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f26\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f26_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='26 Attr. sector/Schwefel', filename=filename)

def ecdf_f27_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f27, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f27_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='27 Attr. sector/Gallagher 101', filename=filename)

def ecdf_f27_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f27\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f27_V_' + str(algorithName)
    plot_ecdf(ecdf_dic,  ptext=ptext, ptitle='27 Attr. sector/Gallagher 101', filename=filename)

def ecdf_f28_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f28, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f28_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='28 Rosenbrock/Rosenbrock', filename=filename)

def ecdf_f28_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f28\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f28_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='28 Rosenbrock/Rosenbrock', filename=filename)

def ecdf_f29_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f29, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f29_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='29 Rosenbrock/Sharp ridge', filename=filename)

def ecdf_f29_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f29\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f29_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='29 Rosenbrock/Sharp ridge', filename=filename)

def ecdf_f30_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f30, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f30_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='30 Rosenbrock/Diff. Powers', filename=filename)

def ecdf_f30_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f30\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f30_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='30 Rosenbrock/Diff. Powers', filename=filename)

def ecdf_f31_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f31, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f31_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='31 Rosenbrock/Rastrigin', filename=filename)

def ecdf_f31_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f31\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f31_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='31 Rosenbrock/Rastrigin', filename=filename)

def ecdf_f32_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f32, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f32_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='32 Rosenbrock/Schaffer F7', filename=filename)

def ecdf_f32_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f32\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f32_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='32 Rosenbrock/Schaffer F7', filename=filename)

def ecdf_f33_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f33, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f33_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='33 Rosenbrock/Schwefel', filename=filename)

def ecdf_f33_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f33\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f33_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='33 Rosenbrock/Schwefel', filename=filename)

def ecdf_f34_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f34, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f34_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='34 Rosenbrock/Gallagher 101', filename=filename)

def ecdf_f34_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f34\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f34_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='34 Rosenbrock/Gallagher 101', filename=filename)

def ecdf_f35_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f35, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f35_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='35 Sharp ridge/Sharp ridge', filename=filename)

def ecdf_f35_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f35\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f35_V' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='35 Sharp ridge/Sharp ridge', filename=filename)

def ecdf_f36_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f36, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f36_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='36 Sharp ridge/Diff. Powers', filename=filename)

def ecdf_f36_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f36\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f36_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='36 Sharp ridge/Diff. Powers', filename=filename)

def ecdf_f37_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f37, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f37_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='37 Sharp ridge/Rastrigin', filename=filename)

def ecdf_f37_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f37\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f37_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='37 Sharp ridge/Rastrigin', filename=filename)

def ecdf_f38_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f38, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f38_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='38 Sharp ridge/Schaffer F7', filename=filename)

def ecdf_f38_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f38\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f38_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='38 Sharp ridge/Schaffer F7', filename=filename)

def ecdf_f39_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f39, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f39_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='39 Sharp ridge/Schwefel', filename=filename)

def ecdf_f39_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f39\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f39_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='39 Sharp ridge/Schwefel', filename=filename)

def ecdf_f40_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f40, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f40_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='40 Sharp ridge/Gallagher 101', filename=filename)

def ecdf_f40_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f40\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f40_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='40 Sharp ridge/Gallagher 101', filename=filename)

def ecdf_f41_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f41, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f41_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='41 Diff. Powers/Diff. Powers', filename=filename)

def ecdf_f41_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f41\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f41_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='41 Diff. Powers/Diff. Powers', filename=filename)

def ecdf_f42_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f42, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f42_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='42 Diff. Powers/Rastrigin', filename=filename)

def ecdf_f42_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f42\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f42_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='42 Diff. Powers/Rastrigin', filename=filename)

def ecdf_f43_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f43, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f43_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='43 Diff. Powers/Schaffer F7', filename=filename)

def ecdf_f43_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f43\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f43_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='43 Diff. Powers/Schaffer F7', filename=filename)

def ecdf_f44_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f44, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f44_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='44 Diff. Powers/Schwefel', filename=filename)

def ecdf_f44_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f44\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f44_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='44 Diff. Powers/Schwefel', filename=filename)

def ecdf_f45_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f45, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f45_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='45 Diff. Powers/Gallagher 101', filename=filename)

def ecdf_f45_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f45\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f45_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='45 Diff. Powers/Gallagher 101', filename=filename)

def ecdf_f46_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f46, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f46_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='46 Rastrigin/Rastrigin', filename=filename)

def ecdf_f46_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f46\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f46_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='46 Rastrigin/Rastrigin', filename=filename)

def ecdf_f47_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f47, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f47_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='47 Rastrigin/Schaffer F7', filename=filename)

def ecdf_f47_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f47\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f47_V' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='47 Rastrigin/Schaffer F7', filename=filename)

def ecdf_f48_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f48, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f48_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='48 Rastrigin/Schwefel', filename=filename)

def ecdf_f48_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f48\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f48_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='48 Rastrigin/Schwefel', filename=filename)

def ecdf_f49_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f49, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f49_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='49 Rastrigin/Gallagher 101', filename=filename)

def ecdf_f49_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f49\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f49_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='49 Rastrigin/Gallagher 101', filename=filename)

def ecdf_f50_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f50, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f50_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='50 Schaffer F7/Schaffer F7', filename=filename)

def ecdf_f50_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f50\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f50_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='50 Schaffer F7/Schaffer F7', filename=filename)

def ecdf_f51_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f51, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f51_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='51 Schaffer F7/Schwefel', filename=filename)

def ecdf_f51_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f51\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f51_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='51 Schaffer F7/Schwefel', filename=filename)

def ecdf_f52_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f52, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f52_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='52 Schaffer F7/Gallagh. 101', filename=filename)

def ecdf_f52_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f52\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f52_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='52 Schaffer F7/Gallagh. 101', filename=filename)

def ecdf_f53_plot(ecdf_dic, nInputs, plot_ecdf, targets_msge):
    ptext = ('bbob-biobj f53, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f53_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='53 Schwefel/Schwefel', filename=filename)

def ecdf_f53_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msge):
    ptext = ('bbob-biobj f53\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f53_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='53 Schwefel/Schwefel', filename=filename)

def ecdf_f54_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f54, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f54_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='54 Schwefel/Gallagher 101', filename=filename)

def ecdf_f54_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f54\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f54_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='54 Schwefel/Gallagher 101', filename=filename)

def ecdf_f55_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f55, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'individual_plots/BBOB-BIOBJ_ECDF_f55_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='55 Gallagher 101/Gallagh. 101', filename=filename)

def ecdf_f55_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f55\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/individual_plots/BBOB-BIOBJ_ECDF_f55_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='55 Gallagher 101/Gallagh. 101', filename=filename)

In [ ]:
def ecdf_all_plot(ecdf_dic, nInputs, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f1-f55, ' + str(nInputs) + '-D\n' + targets_msg() + '\n' + '15 instances')
    filename = 'all_plots/BBOB-BIOBJ_ECDF_all_D' + str(nInputs)
    plot_ecdf(ecdf_dic, nInputs, ptext=ptext, ptitle='', filename=filename)

def ecdf_all_da_plot(ecdf_dic, algorithName, plot_ecdf, targets_msg):
    ptext = ('bbob-biobj f1-f55\n' + targets_msg() + '\n' + '15 instances')
    filename = 'scalability/all_plots/BBOB-BIOBJ_ECDF_all_V_' + str(algorithName)
    plot_ecdf(ecdf_dic, ptext=ptext, ptitle='', filename=filename)

In [ ]:
font = {'family' : 'normal',
        'weight' : 'normal',
        'size'   : 18}
mpl.rc('font', **font)

# ECDF plot function
def plot_ecdf_base(ecdf_dic, nInputs, ptext, ptitle, filename):
    algorithmNames = list(ecdf_dic.keys())
    doe = 11 * nInputs - 1

    plt.rcParams['text.usetex'] = True
    fig, ax = plt.subplots()
    for algoName in algorithmNames:
        ecdf = ecdf_dic[algoName]
        r = [i+1 for i in range(len(ecdf))]
        plt.plot(r, ecdf, label=algoName)
    ax.axvline(x = doe, color = 'k', label = 'DoE', linestyle='dashed')
    ax.set_xscale('log')
    ax.set_xlim([1, budget])
    ax.set_xticks([1, 10, 100, budget])
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_ylim([0, 1])
    if nInputs == 2:
        plt.legend(loc='lower right')
    else:
        plt.legend(loc='center left')
    ax.set_title(ptitle, fontsize=28, weight='bold')
    plt.text(.01, 0.99, ptext, ha='left', va='top', transform=ax.transAxes)
    plt.grid()
    ax.set_ylabel('Fraction of function,target pairs', fontsize=22)
    ax.set_xlabel('\# f-eval', fontsize=22)
    plt.savefig('bbob_plots/' + filename + '_base.png', bbox_inches='tight')
    plt.savefig('bbob_plots/' + filename + '_base.pdf', bbox_inches='tight')
    # plt.show()
    plt.close()

# ECDF plot function
def plot_ecdf_norm(ecdf_dic, nInputs, ptext, ptitle, filename):
    algorithmNames = list(ecdf_dic.keys())
    doe = math.log10((11 * nInputs - 1)/nInputs)

    plt.rcParams['text.usetex'] = True
    fig, ax = plt.subplots()
    for algoName in algorithmNames:
        ecdf = ecdf_dic[algoName]
        r = [math.log10((i+1)/nInputs) for i in range(len(ecdf))]
        plt.plot(r, ecdf, label=algoName, clip_on=False)
    ax.axvline(x = doe, color = 'k', label = 'DoE', linestyle='dashed')
    if (nInputs==2) or (nInputs==3):
        xmax = 3
    elif nInputs==5:
        xmax = 2.5
    elif (nInputs==10) or (nInputs==20):
        xmax = 2
    else:
        xmin, xmax = ax.get_xlim()
    ax.set_xlim([0, xmax])
    ax.set_ylim([0, 1])
    if nInputs == 2:
        plt.legend(loc='lower right')
    else:
        plt.legend(loc='center left')
    plt.title(ptitle, fontsize=28, weight='bold')
    plt.text(.01, 0.99, ptext, ha='left', va='top', transform=ax.transAxes)
    plt.grid()
    plt.ylabel('Fraction of function,target pairs', fontsize=22)
    plt.xlabel('log10(\# f-eval / dimension)', fontsize=22)
    plt.savefig('bbob_plots/' + filename + '_norm.png', bbox_inches='tight')
    plt.savefig('bbob_plots/' + filename + '_norm.pdf', bbox_inches='tight')
    # plt.show()
    plt.close()

# ECDF plot function
def plot_ecdf_da_base(ecdf_dic, ptext, ptitle, filename):
    nInputsList = list(ecdf_dic.keys())
    
    plt.rcParams['text.usetex'] = True
    fig, ax = plt.subplots()
    for dimension in nInputsList:
        ecdf = ecdf_dic[dimension]
        r = [i+1 for i in range(len(ecdf))]
        plt.plot(r, ecdf, label=str(dimension) + '-D')
    ax.set_xscale('log')
    ax.set_xlim([1, budget])
    ax.set_xticks([1, 10, 100, budget])
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_ylim([0, 1])
    plt.legend(loc='lower left')
    plt.title(ptitle, fontsize=28, weight='bold')
    plt.text(.01, 0.99, ptext, ha='left', va='top', transform=ax.transAxes)
    plt.grid()
    plt.ylabel('Fraction of function,target pairs', fontsize=22)
    plt.xlabel('\# f-eval', fontsize=22)
    plt.savefig('bbob_plots/' + filename + '_base.png', bbox_inches='tight')
    plt.savefig('bbob_plots/' + filename + '_base.pdf', bbox_inches='tight')
    # plt.show()
    plt.close()

# ECDF plot function
def plot_ecdf_da_norm(ecdf_dic, ptext, ptitle, filename):
    nInputsList = list(ecdf_dic.keys())    
    plt.rcParams['text.usetex'] = True
    fig, ax = plt.subplots()
    for dimension in nInputsList:
        ecdf = ecdf_dic[dimension]
        r = [math.log10((i+1)/dimension) for i in range(len(ecdf))]
        plt.plot(r, ecdf, label=str(dimension) + '-D')
    nInputs = min(nInputsList)
    if (nInputs==2) or (nInputs==3):
        xmax = 3
    elif nInputs==5:
        xmax = 2.5
    elif (nInputs==10) or (nInputs==20):
        xmax = 2
    else:
        xmin, xmax = ax.get_xlim()
    ax.set_xlim([0, xmax])
    ax.set_ylim([0, 1])
    plt.legend(loc='lower right')
    plt.title(ptitle, fontsize=28, weight='bold')
    plt.text(.01, 0.99, ptext, ha='left', va='top', transform=ax.transAxes)
    plt.grid()
    plt.ylabel('Fraction of function,target pairs', fontsize=22)
    plt.xlabel('log10(\# f-eval / dimension)', fontsize=22)
    plt.savefig('bbob_plots/' + filename + '_norm.png', bbox_inches='tight')
    plt.savefig('bbob_plots/' + filename + '_norm.pdf', bbox_inches='tight')
    # plt.show()
    plt.close()

# Define targets for ECDFs

In [ ]:
# Define the targets function, including the precision targets
# target_func = partial(targets_besthv, precision_targets_np=precision_targets_ec_all())
my_targets = Targets()
my_targets.change_precision_targets(PrecisionTargetsType.BUDGET)
targets_func = my_targets.targets
targets_msg = my_targets.targets_msg
targets_func(35, 2, 1)

# Generate ECDFs

In [ ]:
# Generate ECDF for group plots
ecdf_group_dic = {}
groups_list = ['ss', 'smo', 'si', 'smu', 'sw', 'momo', 'moi', 'momu', 'mow', 'ii', 'imu', 'iw', 'mumu', 'muw', 'ww']

for algorithmName in algorithmNameList:
    for dimensions in dimensionsList:
        for groupName in groups_list:
            file_ecdf = os.path.join(ecdf_data_path, 'ecdf_' + groupName + '_' + algorithmName + '_D' + str(dimensions) + '_' + my_targets.prevision_targets_str() + '.csv')
            if read_ecdf_data and os.path.exists(file_ecdf):
                ecdf = np.genfromtxt(file_ecdf, delimiter = ',')
            else:
                ecdf = locals()['ecdf_' + groupName + '_functions'](dimensions, algorithmName, targets_func)
                ecdf.tofile(file_ecdf, sep = ',')
            ecdf_group_dic[algorithmName + '_D' + str(dimensions) + '_group_' + groupName] = ecdf

In [ ]:
# Generate ECDF for individual plots
ecdf_individual_dic = {}
for algorithmName in algorithmNameList:
    for dimensions in dimensionsList:
        for functionID in range(1, 56):
            file_ecdf = os.path.join(ecdf_data_path, 'ecdf_F' + str(functionID) + '_' + algorithmName + '_D' + str(dimensions) + '_' +  my_targets.prevision_targets_str() + '.csv')
            if read_ecdf_data and os.path.exists(file_ecdf):
                ecdf = np.genfromtxt(file_ecdf, delimiter = ',')
            else:
                ecdf = ecdf_individual_function(functionID, dimensions, algorithmName, targets_func)
                ecdf.tofile(file_ecdf, sep = ',')
            ecdf_individual_dic[algorithmName + '_D' + str(dimensions) + '_F' + str(functionID)] = ecdf

In [ ]:
# Generate ECDF for all plots
ecdf_all_dic = {}
for algorithmName in algorithmNameList:
    for dimensions in dimensionsList:
        file_ecdf = os.path.join(ecdf_data_path, 'ecdf_all_' + algorithmName + '_D' + str(dimensions) + '_' + my_targets.prevision_targets_str() + '.csv')
        if read_ecdf_data and os.path.exists(file_ecdf):
            ecdf = np.genfromtxt(file_ecdf, delimiter = ',')
        else:
            ecdf = ecdf_all_functions(dimensions, algorithmName, targets_func)
            ecdf.tofile(file_ecdf, sep = ',')
        ecdf_all_dic[algorithmName + '_D' + str(dimensions)] = ecdf

# Generate plots

## Normal plots

In [ ]:
if not os.path.exists(ecdf_plot_path):
    os.makedirs(ecdf_plot_path)

In [ ]:
ecdf_plot_path_groups = os.path.join(ecdf_plot_path, 'group_plots')
if not os.path.exists(ecdf_plot_path_groups):
    os.makedirs(ecdf_plot_path_groups)

# Generate all group plots
for dimensions in dimensionsList:
    for groupName in groups_list:
        ecdf_dic = {algorithmName : ecdf_group_dic[algorithmName + '_D' + str(dimensions) + '_group_' + groupName] for algorithmName in algorithmNameList}
        locals()['ecdf_' + groupName + '_plot'](ecdf_dic, dimensions, plot_ecdf_base, targets_msg)
        # locals()['ecdf_' + groupName + '_plot'](ecdf_dic, dimensions, plot_ecdf_norm, targets_msg)

In [ ]:
ecdf_plot_path_individual = os.path.join(ecdf_plot_path, 'individual_plots')
if not os.path.exists(ecdf_plot_path_individual):
    os.makedirs(ecdf_plot_path_individual)

# Generate all individual plots
for dimensions in dimensionsList:
    for functionID in range(1, 56):
        ecdf_dic = {algorithmName : ecdf_individual_dic[algorithmName + '_D' + str(dimensions) + '_F' + str(functionID)] for algorithmName in algorithmNameList}
        locals()['ecdf_f' + str(functionID) + '_plot'](ecdf_dic, dimensions, plot_ecdf_base, targets_msg)
        # locals()['ecdf_f' + str(functionID) + '_plot'](ecdf_dic, dimensions, plot_ecdf_norm, targets_msg)

In [ ]:
ecdf_plot_path_all = os.path.join(ecdf_plot_path, 'all_plots')
if not os.path.exists(ecdf_plot_path_all):
    os.makedirs(ecdf_plot_path_all)

# Generate all all plots
for dimensions in dimensionsList:
    ecdf_dic = {algorithmName : ecdf_all_dic[algorithmName + '_D' + str(dimensions)] for algorithmName in algorithmNameList}
    ecdf_all_plot(ecdf_dic, dimensions, plot_ecdf_base, targets_msg)
    # ecdf_all_plot(ecdf_dic, dimensions, plot_ecdf_norm, targets_msg)

In [ ]:
# groups
# dimensions = 3
# for groupName in groups_list:
#     ecdf_dic = {algorithmName : ecdf_group_dic[algorithmName + '_D' + str(dimensions) + '_group_' + groupName] for algorithmName in algorithmNameList}
#     locals()['ecdf_' + groupName + '_plot'](ecdf_dic, dimensions, plot_ecdf_base, targets_msg)

# all
# for dimensions in dimensionsList:
#     ecdf_dic = {algorithmName : ecdf_all_dic[algorithmName + '_D' + str(dimensions)] for algorithmName in algorithmNameList}
#     ecdf_all_plot(ecdf_dic, dimensions, plot_ecdf_base, targets_msg)

# individual
# dimensions = 20
# for functionID in range(1, 56):
#     ecdf_dic = {algorithmName : ecdf_individual_dic[algorithmName + '_D' + str(dimensions) + '_F' + str(functionID)] for algorithmName in algorithmNameList}
#     locals()['ecdf_f' + str(functionID) + '_plot'](ecdf_dic, dimensions, plot_ecdf_base, targets_msg)

## Scalability w.r.t. decision variables

In [ ]:
ecdf_plot_path_scalability = os.path.join(ecdf_plot_path, 'scalability')
if not os.path.exists(ecdf_plot_path_scalability):
    os.makedirs(ecdf_plot_path_scalability)

In [ ]:
ecdf_plot_path_groups = os.path.join(ecdf_plot_path_scalability, 'group_plots')
if not os.path.exists(ecdf_plot_path_groups):
    os.makedirs(ecdf_plot_path_groups)
    
# Generate all group plots (Scalability w.r.t. decision variables)
for algorithmName in algorithmNameList:
    for groupName in groups_list:
        ecdf_dic = {dimensions : ecdf_group_dic[algorithmName + '_D' + str(dimensions) + '_group_' + groupName] for dimensions in dimensionsList}
        locals()['ecdf_' + groupName + '_da_plot'](ecdf_dic, algorithmName, plot_ecdf_da_norm, targets_msg)

In [ ]:
ecdf_plot_path_individual = os.path.join(ecdf_plot_path_scalability, 'individual_plots')
if not os.path.exists(ecdf_plot_path_individual):
    os.makedirs(ecdf_plot_path_individual)
    
# Generate all individual plots  (Scalability w.r.t. decision variables)
for algorithmName in algorithmNameList:
    for functionID in range(1, 56):
        ecdf_dic = {dimensions : ecdf_individual_dic[algorithmName + '_D' + str(dimensions) + '_F' + str(functionID)] for dimensions in dimensionsList}
        locals()['ecdf_f' + str(functionID) + '_da_plot'](ecdf_dic, algorithmName, plot_ecdf_da_norm, targets_msg)

In [ ]:
ecdf_plot_path_all = os.path.join(ecdf_plot_path_scalability, 'all_plots')
if not os.path.exists(ecdf_plot_path_all):
    os.makedirs(ecdf_plot_path_all)

# Generate all all plots  (Scalability w.r.t. decision variables)
for algorithmName in algorithmNameList:
    ecdf_dic = {dimensions : ecdf_all_dic[algorithmName + '_D' + str(dimensions)] for dimensions in dimensionsList}
    ecdf_all_da_plot(ecdf_dic, algorithmName, plot_ecdf_da_norm, targets_msg)